<a href="https://colab.research.google.com/github/DevzsJhonny/Challenge-AI-ONE---Agente-LOGICAR/blob/dev/Desenv_Agente_LOGICAR_multi_docs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain langchain-community langchain-google-genai langchain-text-splitters pypdf chromadb google-generativeai fpdf

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 790.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# 1. Configuração da API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 2. Carregar TODOS os PDFs que você subiu na pasta principal do Colab
# Ele vai ler automaticamente todos os arquivos que terminam em .pdf
loader = PyPDFDirectoryLoader(".")
documentos = loader.load()

# 3. Processar e dividir os textos em blocos (Chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documentos)

# 4. Criar os Embeddings e o Banco de Dados Vetorial (Chroma)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GEMINI_API_KEY
)
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

# 5. Configurar o Modelo de Linguagem (Gemini Flash)
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

# 6. Definir o Prompt com as novas regras de negócio da Soluções Logicar
template_logicar = """
Você é o assistente virtual oficial da Soluções Logicar.
Sua base de conhecimento é composta por múltiplos manuais (Logística, Frota e Políticas).
Use APENAS o contexto fornecido para responder.

DIRETRIZES DE RESPOSTA:
1. CNH: Informe que é obrigatória a CNH DEFINITIVA. Proibido PPD (provisória).
2. MANUTENÇÃO: Revisão a cada 10.000km ou 1 vez por mês (vistoria preventiva).
3. AVARIAS: O veículo deve ser entregue sem danos; avarias serão cobradas.
4. FRETE: O valor é calculado com base na quilometragem (KM) e tipo de veículo.
5. TIPOS DE VEÍCULOS: Mencione motos, vans, carretos ou carretas conforme o manual de logística.

CONTEXTO:
{context}

PERGUNTA:
{question}

RESPOSTA:"""

PROMPT = PromptTemplate(template=template_logicar, input_variables=["context", "question"])

# 7. Criar o Agente de Respostas (RAG)
agente_logicar = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs={"prompt": PROMPT}
)

print(f"✅ Agente Soluções Logicar pronto! {len(documentos)} páginas de manuais carregadas.")

✅ Agente Soluções Logicar pronto! 3 páginas de manuais carregadas.


#Testando agente

In [ ]:
print("🤖 Sistema Soluções Logicar - Teste de Multi-Documentos")
print("Digite sua pergunta ou 'sair' para encerrar.")

while True:
    pergunta = input("\nSua pergunta: ")
    if pergunta.lower() in ['sair', 'parar', 'exit']: break

    try:
        resposta = agente_logicar.invoke(pergunta)
        print(f"\nAgente: {resposta['result']}")
    except Exception as e:
        print(f"Erro: {e}")

🤖 Sistema Soluções Logicar - Teste de Multi-Documentos
Digite sua pergunta ou 'sair' para encerrar.

Sua pergunta: preciso de CNH para pegar uma moto?

Agente: Olá! Sim, para qualquer veículo da Soluções Logicar, incluindo motos, é **obrigatória a apresentação da CNH Definitiva**. 

Vale ressaltar que:
* **Proibido PPD:** Não aceitamos CNH em período de permissão (provisória).
* **Idade mínima:** O condutor deve ter no mínimo 21 anos completos.


KeyboardInterrupt: Interrupted by user